# Engineering Lab — v0.1 (Fundacion)

Alcance de esta version: configuracion, descarga del modelo, clonacion del repositorio, chat con el modelo y visualizacion del estado.

No hay parser, index, graph, retriever, planner, agent ni patcher todavia — eso llega en v0.2 en adelante. Este notebook es deliberadamente chico.

## 0. Montar Drive (opcional para probar, recomendado para persistir)

Esto es lo que permite que `history/`, `config.json` y el modelo descargado sobrevivan a que se caiga la sesion de Colab.

Si el montaje falla (error `mount failed`), es casi siempre el navegador bloqueando el popup de autorizacion de Google, o una sesion de Drive colgada de un intento anterior. Nada del codigo del laboratorio depende de que esto funcione a la primera: si falla, todo cae a rutas locales dentro de la sesion (`./config.json`, `./history`, `./models`), que simplemente no sobreviven un reinicio del runtime. Podes seguir probando sin Drive y montarlo despues.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except Exception as e:
    print("No se pudo montar Drive, sigo sin persistencia:", e)
    print("Reintentar: Entorno de ejecucion > Desconectar y eliminar entorno de ejecucion, y volver a correr esta celda.")

## 1. Traer el laboratorio a la sesion

"La sesion" es la maquina virtual efimera de Colab (todo lo que esta bajo `/content`, se borra al terminar la sesion). El laboratorio en si (este codigo) tiene que llegar ahi de alguna forma. Dos opciones:

- **Recomendada: clonarlo desde GitHub** (si ya lo subiste a un repo, poné la URL abajo). Asi en cada sesion nueva es una linea, y los cambios que hagas y subas a GitHub se traen con `!git pull` en vez de volver a subir un zip a mano.
- **Manual**: en el panel izquierdo de Colab (icono de carpeta) hay un boton para subir archivos/carpetas directo a `/content`. Si subiste el `.zip`, hay que descomprimirlo con `!unzip nombre.zip -d /content/`.

Dejo las dos opciones en la celda de abajo; comenta la que no uses.

In [ ]:
# Opcion A (recomendada): siempre tomar el repo de GitHub como fuente de verdad,
# descartando cualquier cambio local que haya quedado en la sesion.
ENGINEERING_LAB_REPO = "https://github.com/Gunther-Frager/engineering-lab.git"

import os
if not os.path.exists("/content/engineering-lab"):
    !git clone {ENGINEERING_LAB_REPO} /content/engineering-lab
else:
    !cd /content/engineering-lab && git fetch origin && git reset --hard origin/main

# Opcion B: si subiste el .zip a mano al panel de archivos de la izquierda,
# comenta el bloque de arriba y descomenta esto:
# !unzip -oq /content/engineering-lab-v0.1.zip -d /content/

%cd /content/engineering-lab

## 2. Instalar dependencias

`llama-cpp-python` se compila con soporte CUDA para poder hacer offload de capas a la T4. La instalacion tarda unos minutos la primera vez.

In [ ]:
!pip install -q huggingface_hub gitpython ipywidgets
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

## 3. Configuracion

Unica celda donde se tocan parametros. Todo lo demas los lee de aca.

In [ ]:
from config.config import LabConfig

config = LabConfig(
    REPO_URL="https://github.com/Gunther-Frager/bannercito.git",
    MODEL="qwen2.5-coder-14b-instruct",
    GPU="T4",
    TEMPERATURE=0.2,
    MAX_CONTEXT=8192,
)
config.save()
config

## 4. Clonar el repositorio a analizar

In [ ]:
from repository.git_repo import GitRepository

repo = GitRepository(config.REPO_URL, local_path="./workspace/repo")
print(repo.clone())
repo_ready = True

## 5. Cargar el modelo

Primera vez: descarga el GGUF cuantizado (~8-10GB) y lo cachea en Drive. Las siguientes veces solo carga.

In [ ]:
from llm.qwen import QwenCoder

llm = QwenCoder(n_ctx=config.MAX_CONTEXT)
model_ready = True

## 6. Estado del laboratorio

In [ ]:
from ui.status import render_status

render_status(config, repo_ready=repo_ready, model_ready=model_ready)

## 7. Chat

Cada intercambio queda logueado en `history/NNNNN/` (prompt, respuesta, config usada). Eso es lo unico que en v0.1 se acerca al principio de "todo debe ser observable": el registro existe, aunque todavia no hay planner/agent que expliquen por que abrieron tal archivo — eso es v0.4 en adelante.

In [ ]:
from history.logger import HistoryLogger

history = HistoryLogger()

def chat(prompt: str) -> str:
    response = llm.generate(prompt, temperature=config.TEMPERATURE)
    entry_id = history.log(prompt, response, config.__dict__)
    print(f"[history/{entry_id}]\n")
    print(response)
    return response

In [ ]:
chat("Decime en 3 lineas que hace este repo, basandote en su README si existe.")